# Task-category robustness figure

Robustness-tier check (CLAUDE.md, "Task categories") on claim 2: does the
within-task > between-task ordering hold when "task" means the *category* of
paradigm — `naturalistic`, `taskscapes` (systematic stimulus-space
exploration: emotion videos, triplets, multfs, things) or `localizers`
(hcptrt's eight tasks incl. rest, floc, language localizers, retinotopy) —
rather than the dataset? Reads `output_data/group_stats/category_cross_context.tsv`
and plots only.

**Ungated (`gate="all"`) is the primary view**: the 1800 s usable-duration gate
removes `things`, `triplets`, `floc`, `langlocalizer` and `retinotopy`
entirely, leaving one dataset each for taskscapes (`multfs`) and localizers
(`hcptrt`). The duration confound is large ungated (see
`category_duration_balance.tsv`), so report it beside this panel.

**Standalone figure, deliberately not placed in `connectome_figure.svg`.**

Panel: `category_bins.png` — the four subject x category bins, one group of
bars per network. Hue marks category (within/between), alpha marks subject
(within/between).


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from airoh.figures import panel_size

FIGURE_DPI = int(os.environ.get("FIGURE_MONTAGE_DPI", 300))

output_dir = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data")).resolve()
figures_base = Path(os.environ.get("FIGURES_DIR", output_dir / "figures")).resolve()

project_root = output_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

figure_dir = figures_base / "figure_categories"
figure_dir.mkdir(parents=True, exist_ok=True)

with open(project_root / "invoke.yaml") as handle:
    invoke_config = yaml.safe_load(handle)

PARCELLATION = invoke_config["parcellation"]
NETWORK_ORDER = invoke_config["parcellations"][PARCELLATION]["network_order"]

category_path = output_dir / "group_stats" / "category_cross_context.tsv"
category_bins = (
    pd.read_csv(category_path, sep="\t") if category_path.exists() else pd.DataFrame()
)
print(f"📂 {PARCELLATION}: {len(category_bins)} category-bin rows")


📂 cneuromod2026: 180 category-bin rows


In [2]:
CATEGORY_HUE = {"within-category": "#2a78d6", "between-category": "#eb6834"}
SUBJECT_ALPHA = {"within-subject": 1.0, "between-subject": 0.45}
BIN_ORDER = [
    f"{s} / {c}" for c in CATEGORY_HUE for s in SUBJECT_ALPHA
]

figsize = panel_size("figure_categories/category_bins.png", (6.0, 4.5))
fig, ax = plt.subplots(figsize=figsize, layout="constrained")
ungated = category_bins[category_bins["gate"] == "all"] if len(category_bins) else category_bins

legend_handles, legend_labels = [], []
if len(ungated):
    x = np.arange(len(NETWORK_ORDER))
    width = 0.2
    for i, bin_label in enumerate(BIN_ORDER):
        subject_part, category_part = bin_label.split(" / ")
        values = []
        for network in NETWORK_ORDER:
            row = ungated[(ungated["network"] == network) & (ungated["bin"] == bin_label)]
            values.append(row["median"].iloc[0] if len(row) else np.nan)
        bars = ax.bar(x + (i - 1.5) * width, values, width, color=CATEGORY_HUE[category_part],
                      alpha=SUBJECT_ALPHA[subject_part], edgecolor="white", linewidth=0.4)
        legend_handles.append(bars[0])
        legend_labels.append(bin_label)
    ax.set_xticks(x)
    ax.set_xticklabels(NETWORK_ORDER, rotation=45, ha="right", fontsize=7)
    ax.set_ylabel("median similarity (Fisher-z)")
    ax.spines[["top", "right"]].set_visible(False)
else:
    ax.text(0.5, 0.5, "fewer than two task categories present\n(smoke run)",
            ha="center", va="center", transform=ax.transAxes, color="0.5", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

fig.savefig(figure_dir / "category_bins.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'category_bins.png'} at {figsize} in")

if legend_handles:
    legend_size = panel_size("figure_categories/category_bins_legend.png", (6.0, 0.9))
    legend_fig = plt.figure(figsize=legend_size, layout="constrained")
    legend_fig.legend(legend_handles, legend_labels, loc="center", ncol=2, fontsize=7,
                      frameon=False)
    legend_fig.savefig(figure_dir / "category_bins_legend.png", dpi=FIGURE_DPI)
    plt.close(legend_fig)


✅ wrote /home/pbellec/git/cneuromod.all.connectome_stats/output_data/figures/figure_categories/category_bins.png at (6.0, 4.5) in
